# OTT Streaming Content Trend Analysis — Netflix Catalog

**Objective:** Explore genre mix, content ratings, release/addition trends, regional mix and runtime to support content-acquisition decisions.

**Dataset:** Netflix Movies and TV Shows catalog snapshot (TidyTuesday, 2021-04-20). This is a single-platform catalog snapshot, not viewing/engagement data and not a panel of Netflix's historical catalogs. It supports catalog composition analysis, not claims about audience demand or retention.

**Run:** Execute all cells from top to bottom. The notebook downloads the public CSV if it is not already in `../data/`. Internet access is required for the first run.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
REPORT_DIR = ROOT / "reports"
DATA_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-04-20/netflix_titles.csv"
RAW_PATH = DATA_DIR / "netflix_titles.csv"

if not RAW_PATH.exists():
    df_raw = pd.read_csv(DATA_URL, encoding="utf-8")
    df_raw.to_csv(RAW_PATH, index=False)
else:
    df_raw = pd.read_csv(RAW_PATH, encoding="utf-8")

print(f"Raw shape: {df_raw.shape}")
display(df_raw.head())


## 1. Data quality and cleaning

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower()
df = df.drop_duplicates(subset=["show_id"]).copy()

# Preserve missing categorical information explicitly; do not discard titles.
for col in ["director", "cast", "country", "rating", "duration", "listed_in"]:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()

df["date_added"] = pd.to_datetime(df["date_added"].astype(str).str.strip(), errors="coerce")
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int64")
df["date_added_year"] = df["date_added"].dt.year
df["date_added_month"] = df["date_added"].dt.month
df["date_added_month_name"] = df["date_added"].dt.month_name()

# Parse duration into a numeric measure and unit. Movie runtime and TV seasons remain separate.
duration_parts = df["duration"].str.extract(r"(?P<duration_value>\d+)")
df["duration_value"] = pd.to_numeric(duration_parts["duration_value"], errors="coerce")
df["duration_unit"] = np.select(
    [df["duration"].str.contains("min", case=False, na=False),
     df["duration"].str.contains("season", case=False, na=False)],
    ["minutes", "seasons"], default="unknown"
)

df["country_list"] = df["country"].apply(lambda x: [] if x == "Unknown" else [v.strip() for v in x.split(",") if v.strip()])
df["genre_list"] = df["listed_in"].apply(lambda x: [] if x == "Unknown" else [v.strip() for v in x.split(",") if v.strip()])
df["primary_country"] = df["country_list"].apply(lambda x: x[0] if x else "Unknown")

genres = df[["show_id", "genre_list"]].explode("genre_list").rename(columns={"genre_list":"genre"})
genres = genres.dropna()
countries = df[["show_id", "country_list"]].explode("country_list").rename(columns={"country_list":"country"})
countries = countries.dropna()

quality = pd.DataFrame({
    "metric": ["raw_rows", "deduplicated_rows", "duplicate_rows_removed", "missing_date_added", "missing_release_year"],
    "value": [len(df_raw), len(df), len(df_raw)-len(df), int(df["date_added"].isna().sum()), int(df["release_year"].isna().sum())]
})
display(quality)
display(df.head())


## 2. Catalog overview

In [ ]:
type_counts = df["type"].value_counts()
display(type_counts.to_frame("titles"))
print("Unique titles:", df["show_id"].nunique())
print("Countries (expanded):", countries["country"].nunique())
print("Genres (expanded):", genres["genre"].nunique())


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
type_counts.plot(kind="bar", ax=ax)
ax.set(title="Catalog composition by content type", xlabel="", ylabel="Number of titles")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 3. Genre popularity

In [ ]:
genre_counts = genres["genre"].value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(9,6))
genre_counts.plot(kind="barh", ax=ax)
ax.set(title="Top 15 listed genres", xlabel="Title-genre associations", ylabel="")
plt.tight_layout()
plt.show()


## 4. Content rating mix

In [ ]:
rating_counts = df["rating"].replace("Unknown", np.nan).value_counts().head(12).sort_values()
fig, ax = plt.subplots(figsize=(9,5))
rating_counts.plot(kind="barh", ax=ax)
ax.set(title="Most common content ratings", xlabel="Number of titles", ylabel="")
plt.tight_layout()
plt.show()


## 5. Release-year trends

In [ ]:
release_counts = df.dropna(subset=["release_year"]).groupby("release_year").size()
release_counts = release_counts.loc[release_counts.index >= 2000]
fig, ax = plt.subplots(figsize=(10,4))
release_counts.plot(ax=ax)
ax.set(title="Catalog titles by original release year (2000 onward)", xlabel="Release year", ylabel="Titles in snapshot")
plt.tight_layout()
plt.show()


## 6. Content addition trends

In [ ]:
added_counts = df.dropna(subset=["date_added"]).groupby("date_added_year").size()
fig, ax = plt.subplots(figsize=(10,4))
added_counts.plot(kind="bar", ax=ax)
ax.set(title="Titles by year added to Netflix (available date_added values)", xlabel="Year added", ylabel="Titles")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 7. Regional content mix

In [ ]:
country_counts = countries["country"].value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(9,6))
country_counts.plot(kind="barh", ax=ax)
ax.set(title="Top 15 production countries (multi-country titles counted for each)", xlabel="Title-country associations", ylabel="")
plt.tight_layout()
plt.show()


## 8. Movie runtime distribution

In [ ]:
movie_runtime = df.loc[(df["type"] == "Movie") & (df["duration_unit"] == "minutes"), "duration_value"].dropna()
fig, ax = plt.subplots(figsize=(9,4))
sns.histplot(movie_runtime, bins=30, ax=ax)
ax.set(title="Movie runtime distribution", xlabel="Runtime (minutes)", ylabel="Number of movies")
plt.tight_layout()
plt.show()
print("Movie runtime median:", round(movie_runtime.median(), 1), "minutes")


## 9. Comparative cuts for strategy

In [ ]:
# Genre by content type: top genres overall, shown as title counts by type.
top_genres = genres["genre"].value_counts().head(10).index
genre_type = (df[["show_id", "type"]].merge(genres, on="show_id")
              .query("genre in @top_genres")
              .groupby(["genre", "type"]).size().unstack(fill_value=0))
display(genre_type.sort_values(by=list(genre_type.columns), ascending=False))

# Country-level genre mix can be explored by joining the exploded bridge tables.
country_genre = (countries.merge(genres, on="show_id")
                 .groupby(["country", "genre"]).size().rename("title_count").reset_index())
display(country_genre.sort_values("title_count", ascending=False).head(20))


## 10. Export cleaned tables for SQL and generate the strategy memo

In [ ]:
# Export a clean title table and normalized bridge tables for SQL.
title_export = df.drop(columns=["country_list", "genre_list"]).copy()
title_export["date_added"] = title_export["date_added"].dt.strftime("%Y-%m-%d")
title_export.to_csv(DATA_DIR / "netflix_titles_clean.csv", index=False)
genres.to_csv(DATA_DIR / "title_genres.csv", index=False)
countries.to_csv(DATA_DIR / "title_countries.csv", index=False)

# Data-backed metrics used in the memo. These are descriptive catalog metrics, not audience metrics.
movie_share = (df["type"].eq("Movie").mean() * 100)
tv_share = (df["type"].eq("TV Show").mean() * 100)
top_genre = genres["genre"].value_counts().idxmax() if len(genres) else "Not available"
top_genre_n = int(genres["genre"].value_counts().max()) if len(genres) else 0
top_country = countries["country"].value_counts().idxmax() if len(countries) else "Not available"
top_country_n = int(countries["country"].value_counts().max()) if len(countries) else 0
median_runtime = float(movie_runtime.median()) if len(movie_runtime) else float("nan")
peak_add_year = int(added_counts.idxmax()) if len(added_counts) else None
peak_add_n = int(added_counts.max()) if len(added_counts) else 0
rating_top = df.loc[df["rating"] != "Unknown", "rating"].value_counts()
top_rating = rating_top.index[0] if len(rating_top) else "Not available"
top_rating_n = int(rating_top.iloc[0]) if len(rating_top) else 0

memo = f'''# Content Strategy Memo — Netflix Catalog Snapshot

**To:** Content Acquisition & Strategy Team  
**From:** Data Analysis  
**Subject:** Catalog mix, genre concentration and regional opportunity  
**Dataset:** Netflix Movies and TV Shows catalog snapshot (TidyTuesday, 2021-04-20)  
**Scope note:** This is a catalog inventory, not viewing, completion, subscriber or revenue data. Findings describe what is listed, not what audiences watched or preferred.

## Executive readout

The cleaned snapshot contains **{len(df):,} unique titles**. Movies represent **{movie_share:.1f}%** and TV Shows **{tv_share:.1f}%** of the catalog. The most frequently listed genre is **{top_genre}** ({top_genre_n:,} title-genre associations); the most represented production country is **{top_country}** ({top_country_n:,} title-country associations). The median movie runtime is **{median_runtime:.0f} minutes**.

## Three actionable recommendations

1. **Protect breadth while auditing genre concentration.**  
   {top_genre} is the most common listed genre in this snapshot. Treat this as evidence of catalog supply concentration, not proof of demand. Before approving additional titles in the same category, compare genre-level viewing hours, completion and acquisition cost. Set a portfolio target that preserves a mix of established genres and underrepresented categories.

2. **Build a regional acquisition shortlist rather than relying on country volume alone.**  
   {top_country} has the largest number of title-country associations. Multi-country titles are counted once for each listed country, so these are associations, not exclusive national totals. Identify markets with low catalog representation, then evaluate local-language audience demand, rights availability and cost per expected viewing hour before commissioning or licensing.

3. **Use format and release-window segmentation in the next slate review.**  
   Movies account for {movie_share:.1f}% of titles, while TV Shows account for {tv_share:.1f}%. The median movie runtime is {median_runtime:.0f} minutes. Compare movie and series performance separately, and test whether underserved runtime/season-length bands align with audience behavior. The peak year for additions with a recorded date is {peak_add_year} ({peak_add_n:,} titles); validate this pattern against complete internal ingestion records before interpreting it as a true acquisition cycle.

## What to validate with first-party data

- Viewing hours, unique viewers, completion rate and repeat viewing by genre, country and format.
- Subscriber acquisition/retention contribution and title-level acquisition/production cost.
- Rights windows, local-language availability, marketing spend and release calendar.
- Missingness in country, rating, duration and date-added fields.

## Method and caveats

Duplicate `show_id` rows were removed. Missing descriptive fields were retained as `Unknown`; dates and numeric release years were parsed with invalid values left missing. Genre and country strings were split into bridge tables. A title associated with multiple genres/countries contributes to each relevant category. `date_added` measures the date recorded in this snapshot, not a complete history of catalog changes. This dataset contains no ratings score or audience metrics.
'''
(REPORT_DIR / "content_strategy_memo.md").write_text(memo, encoding="utf-8")
print("Exported clean tables and reports/content_strategy_memo.md")


## 11. Final checklist and limitations

- [x] EDA notebook with 8 visualizations.
- [x] Cleaning and feature engineering documented.
- [x] SQL-ready title, genre and country tables exported.
- [x] Content strategy memo generated from the data when the notebook is run.
- [x] Multi-category counting caveats documented.

**Important:** The source catalog is a historical snapshot. Do not describe this as a current Netflix catalog or use catalog counts as a proxy for popularity. For a true content-performance recommendation, join internal viewership, completion, cost and subscriber data.
